# 03 — Model Training & Evaluation

Runs the bakeoff ladder, evaluates subgroup errors, inspects calibration, and looks at the bakeoff artifacts.

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BASE = pathlib.Path('..')
bakeoff  = json.loads((BASE / 'artifacts/bakeoff/model_comparison.json').read_text())
ablation = json.loads((BASE / 'artifacts/ablation/feature_ablation.json').read_text())
metrics  = json.loads((BASE / 'artifacts/candidate_smoke/metrics.json').read_text())
shap     = json.loads((BASE / 'artifacts/candidate_smoke/shap_summary.json').read_text())
print('Artifacts loaded.')

In [ ]:
# Model ladder summary
models = bakeoff['models']
print(f'{"Model":<28} {"MAE":>8} {"RMSE":>8} {"Coverage":>10} {"Width":>8} {"p95 ms":>8}')
print('-' * 72)
for name, m in models.items():
    short = name[:27]
    cov = m.get('coverage_80', 0)
    width = m.get('mean_width', 0)
    p95 = m.get('p95_ms', 0)
    marker = ' ★' if 'CQR' in name else ''
    print(f'{short:<28} {m["mae"]:>7.3f}s {m["rmse"]:>7.3f}s {cov:>9.1%} {width:>7.2f}s {p95:>7.1f}ms{marker}')

In [ ]:
# Why CatBoost shows Ridge-identical metrics
# It falls back to Ridge when catboost isn't installed
print('CatBoost == Ridge?', abs(models['CatBoostPaceModel']['mae'] - models['RidgeRegressionBaseline']['mae']) < 0.001)
print("\nThis is the catboost import fallback — when catboost isn't installed,")
print('the wrapper falls back to RidgeRegressionBaseline.')

In [ ]:
# Champion model subgroup breakdown
print('Champion model (QuantileLightGBM + CQR):')
print(f'  Overall MAE:   {metrics["mae"]:.4f}s')
print(f'  Overall RMSE:  {metrics["rmse"]:.4f}s')
print(f'  80% Coverage:  {metrics["coverage_80"]:.1%}  (calibrated: {metrics["coverage_80_calibrated"]:.1%})')
print(f'  Mean Width:    {metrics["mean_width"]:.3f}s  (calibrated: {metrics["mean_width_calibrated"]:.3f}s)')
print(f'  CQR q_hat:     n/a (stored in calibrator.json)')
print()
print('Per compound:')
for k, v in metrics.get('per_compound', {}).items():
    flag = '  ← warmup non-linearity' if k == 'HARD' else ''
    print(f'  {k:8s}: {v:.3f}s{flag}')
print()
print('Per stint:')
for k, v in metrics.get('per_stint', {}).items():
    note = '  ← out-lap + fuel load' if '1' in k else ''
    print(f'  {k:10s}: {v:.3f}s{note}')

In [ ]:
# Plot: bakeoff MAE comparison
names  = list(models.keys())
maes   = [models[n]['mae'] for n in names]
shorts = ['LastLap', 'RollingMed(3)', 'Ridge', 'LightGBM', 'CatBoost', 'LGBM+CQR ★']
cols   = ['#8b9bb4'] * 5 + ['#22c55e']

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#080c14'); ax.set_facecolor('#0f172a')
bars = ax.barh(shorts, maes, color=cols, height=0.6, edgecolor='#1e293b')
ax.axvline(maes[-1], color='#22c55e', linewidth=1, linestyle='--', alpha=0.5)
for bar, v in zip(bars, maes):
    ax.text(v + 0.03, bar.get_y() + bar.get_height()/2, f'{v:.3f}s', va='center', color='white', fontsize=9)
ax.set_xlabel('MAE (seconds)', color='#8b9bb4'); ax.tick_params(colors='#8b9bb4')
ax.set_title('Bakeoff — Pace MAE on held-out races', color='white', fontsize=12)
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP: which features actually drive predictions?
shap_sorted = sorted(shap.items(), key=lambda x: x[1], reverse=True)[:12]
feats = [k for k,_ in shap_sorted]
vals  = [v for _,v in shap_sorted]

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#080c14'); ax.set_facecolor('#0f172a')
cols = ['#ff1801' if v > 50 else '#f59e0b' if v > 10 else '#00d2be' for v in vals]
bars = ax.barh(feats[::-1], vals[::-1], color=cols[::-1], height=0.65, edgecolor='#1e293b')
for bar, v in zip(bars, vals[::-1]):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2, f'{v:.1f}', va='center', color='white', fontsize=8)
ax.set_xlabel('Mean |SHAP| value', color='#8b9bb4'); ax.tick_params(colors='#8b9bb4')
ax.set_title('Feature importance (SHAP)', color='white', fontsize=12)
for sp in ax.spines.values(): sp.set_edgecolor('#1e293b')
plt.tight_layout(); plt.show()

In [ ]:
# LOFO ablation: what happens when you remove each feature group
groups = ablation['ranked_by_importance']
base   = ablation['baseline']['mae']
print(f'Baseline MAE (all features): {base:.4f}s')
print()
print(f'{"Group removed":<30} {"Ablated MAE":>12} {"ΔMAE":>10} {"ΔCoverage":>12}')
print('-' * 66)
for g in groups:
    grp = ablation['groups'][g]
    d_mae = grp['delta_mae']
    d_cov = grp['delta_coverage'] * 100
    flag = '  ← CRITICAL' if d_mae > 1 else '  ← moderate' if d_mae > 0.05 else ''
    print(f'{g:<30} {grp["mae_ablated"]:>10.4f}s {d_mae:>+9.4f}s {d_cov:>+10.2f}%{flag}')

In [ ]:
# CQR calibration: raw vs calibrated
raw_cov = metrics['coverage_80']
cal_cov = metrics['coverage_80_calibrated']
raw_w   = metrics['mean_width']
cal_w   = metrics['mean_width_calibrated']

print('Calibration effect (rolling 3-race CQR window):')
print(f'  Coverage: {raw_cov:.1%} → {cal_cov:.1%}  (+{(cal_cov-raw_cov)*100:.1f}pp)')
print(f'  Width:    {raw_w:.3f}s → {cal_w:.3f}s  (+{cal_w-raw_w:.3f}s — price of wider intervals)')
print()
print('Hard compound bias correction (stored in calibrator.json):')
calib_file = pathlib.Path('../artifacts/candidate_smoke/model_quantile/calibrator.json')
if calib_file.exists():
    calib = json.loads(calib_file.read_text())
    for k, v in calib.items():
        print(f'  {k}: {v}')

In [ ]:
# Expanding walk-forward split structure
splits = json.loads(pathlib.Path('../artifacts/candidate_smoke/splits.json').read_text())
print(f'Train races: {len(splits["train"])}')
print(f'Val races:   {len(splits["validation"])}')
print(f'Test races:  {len(splits["test"])}')
print()
print('Test races used for final evaluation:')
for s in splits['test']:
    print(f'  {s}')